In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Set global plotting style parameters for figures
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300

# Section 1: Definition of SIR Model and basic parameters
def sir_derivatives(t, y, beta, gamma, N):
    """
    Calculates the derivatives for the SIR epidemic model.

    Inputs:
      t     : Current time (float)
      y     : Current state vector [S, I, R] (array-like)
      beta  : Transmission rate (float)
      gamma : Recovery rate (float)
      N     : Total population (int or float)

    Outputs:
      np.array : The derivatives [dS/dt, dI/dt, dR/dt]
    """
    S, I, R = y
    dSdt = -beta * S * I / N
    dIdt = beta * S * I / N - gamma * I
    dRdt = gamma * I
    return np.array([dSdt, dIdt, dRdt])

# Baseline Initial Conditions and Parameters
N = 1000        # Total population
S0 = 990        # Initial susceptible population
I0 = 10         # Initial infected population
R0 = 0          # Initial recovered population
y0 = np.array([S0, I0, R0]) # Initial state vector

beta = 1.87     # Transmission rate (Days^-1)
gamma = 0.5     # Recovery rate (Days^-1)
t_span = (0, 30)  # Time domain for the simulation (0 to 30 days)

# Section 2: The Euler Method
def euler_step(f, t, y, h, *args):
    """
    Performs a single step using the Explicit Euler method.

    Inputs:
      f    : Function returning the system of ODEs
      t    : Current time
      y    : Current state vector
      h    : Step size
      args : Additional arguments for the function f (e.g., beta, gamma, N)

    Outputs:
      Updated state vector at time t + h
    """
    return y + h * f(t, y, *args)

def rk4_step(f, t, y, h, *args):
    """
    Performs a single step using the Fourth-Order Runge-Kutta (RK4) method.

    Inputs:
      f    : Function returning the system of ODEs
      t    : Current time
      y    : Current state vector
      h    : Step size
      args : Additional arguments for the function f

    Outputs:
      Updated state vector at time t + h
    """
    k1 = f(t, y, *args)
    k2 = f(t + h/2, y + h/2 * k1, *args)
    k3 = f(t + h/2, y + h/2 * k2, *args)
    k4 = f(t + h, y + h * k3, *args)
    return y + (h / 6) * (k1 + 2*k2 + 2*k3 + k4)

def solve_ode_custom(method, f, t_span, y0, h, *args):
    """
    Iteratively solves an ODE system using a specified numerical step method.

    Inputs:
      method : Function for the numerical step (e.g., euler_step, rk4_step)
      f      : Function returning the system of ODEs
      t_span : Tuple specifying (t_start, t_end)
      y0     : Initial state vector
      h      : Step size
      args   : Additional arguments for the function f

    Outputs:
      t_eval : Array of evaluated time points
      y_eval.T : Array of state vectors evaluated at each time point (transposed)
    """
    t_eval = np.arange(t_span[0], t_span[1] + h, h)
    y_eval = np.zeros((len(t_eval), len(y0)))
    y_eval[0] = y0
    for i in range(1, len(t_eval)):
        y_eval[i] = method(f, t_eval[i-1], y_eval[i-1], h, *args)
    return t_eval, y_eval.T


In [ ]:
# Figure 1: Baseline Simulation using Euler and RK4 with small step size
h_fine = 0.001
t_rk4, y_rk4 = solve_ode_custom(rk4_step, sir_derivatives, t_span, y0, h_fine, beta, gamma, N)
t_euler, y_euler = solve_ode_custom(euler_step, sir_derivatives, t_span, y0, h_fine, beta, gamma, N)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left Figure: Explicit Euler
axes[0].plot(t_euler, y_euler[0], label='Susceptible ($S$)', color='blue', linewidth=2.0, linestyle='--')
axes[0].plot(t_euler, y_euler[1], label='Infected ($I$)', color='red', linewidth=2.0, linestyle='--')
axes[0].plot(t_euler, y_euler[2], label='Recovered ($R$)', color='green', linewidth=2.0, linestyle='--')
axes[0].set_title('Explicit Euler Simulation ($h=0.001$)')
axes[0].set_xlabel('Time (Days)')
axes[0].set_ylabel('Population')
axes[0].legend(loc='center right')

# Right Figure: RK4
axes[1].plot(t_rk4, y_rk4[0], label='Susceptible ($S$)', color='blue', linewidth=2.0)
axes[1].plot(t_rk4, y_rk4[1], label='Infected ($I$)', color='red', linewidth=2.0)
axes[1].plot(t_rk4, y_rk4[2], label='Recovered ($R$)', color='green', linewidth=2.0)
axes[1].set_title('RK4 Simulation ($h=0.001$)')
axes[1].set_xlabel('Time (Days)')
axes[1].legend(loc='center right')

plt.tight_layout()
plt.savefig('Figure1_Baseline.png', bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: Stability Comparison (h=0.001 vs h=1.0)
h_coarse = 1.0

# Calculate coarse step trajectories
t_euler_coarse, y_euler_coarse = solve_ode_custom(euler_step, sir_derivatives, t_span, y0, h_coarse, beta, gamma, N)
t_rk4_coarse, y_rk4_coarse = solve_ode_custom(rk4_step, sir_derivatives, t_span, y0, h_coarse, beta, gamma, N)

fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

# Left Figure: Explicit Euler
axes2[0].plot(t_euler, y_euler[1], label='Reference ($h=0.001$)', color='black', linestyle='--', linewidth=2)
axes2[0].plot(t_euler_coarse, y_euler_coarse[1], label='Coarse Step ($h=1.0$)', color='orange', marker='o', linewidth=2)
axes2[0].set_title('Explicit Euler Stability', fontsize=14)
axes2[0].set_xlabel('Time (Days)')
axes2[0].set_ylabel('Infected Population ($I$)')
axes2[0].legend(loc='upper right')
axes2[0].grid(True, linestyle=':', alpha=0.7)

# Right Figure: RK4
axes2[1].plot(t_rk4, y_rk4[1], label='Reference ($h=0.001$)', color='black', linestyle='--', linewidth=2)
axes2[1].plot(t_rk4_coarse, y_rk4_coarse[1], label='Coarse Step ($h=1.0$)', color='red', marker='s', linewidth=2)
axes2[1].set_title('RK4 Stability', fontsize=14)
axes2[1].set_xlabel('Time (Days)')
axes2[1].legend(loc='upper right')
axes2[1].grid(True, linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig('Figure2_Stability_Comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Global Truncation Error Analysis
target_t = 10.0 # Calculate error near the epidemic peak
I_ref = y_rk4[1][int(target_t / h_fine)] # Extract the exact reference value at t=10

h_vals = np.array([0.1, 0.2, 0.25, 0.4, 0.5, 1.0])
err_euler = []
err_rk4 = []

for h in h_vals:
    # Simulate up to t=10
    _, y_eu_temp = solve_ode_custom(euler_step, sir_derivatives, (0, target_t), y0, h, beta, gamma, N)
    _, y_rk_temp = solve_ode_custom(rk4_step, sir_derivatives, (0, target_t), y0, h, beta, gamma, N)

    # Absolute error = |Simulated - Reference|
    err_euler.append(abs(y_eu_temp[1][-1] - I_ref))
    err_rk4.append(abs(y_rk_temp[1][-1] - I_ref))

plt.figure(figsize=(8, 6))
plt.loglog(h_vals, err_euler, marker='o', color='orange', linewidth=2, label='Explicit Euler Error')
plt.loglog(h_vals, err_rk4, marker='s', color='red', linewidth=2, label='RK4 Error')

# Theoretical slopes
C_euler = err_euler[0] / h_vals[0]
plt.loglog(h_vals, C_euler * h_vals, linestyle='--', color='gray', label='Theoretical Slope $O(h)$')

C_rk4 = err_rk4[0] / (h_vals[0]**4)
plt.loglog(h_vals, C_rk4 * (h_vals**4), linestyle='--', color='blue', alpha=0.5, label='Theoretical Slope $O(h^4)$')

plt.title('Global Truncation Error at $t=10$', fontsize=14)
plt.xlabel('Step Size $h$ (Log Scale)')
plt.ylabel('Absolute Error $|I_{simulated} - I_{ref}|$ (Log Scale)')
plt.legend()
plt.grid(True, which="both", linestyle=':', alpha=0.7)
plt.tight_layout()
plt.savefig('Figure3_Error_Analysis.png', dpi=300, bbox_inches='tight')
plt.show()